# 10 model comparison

## Comparison summary

All reported classification metrics come from saved result JSON files and test predictions; each operating threshold was selected on validation. XGBoost had the highest individual-model precision (0.9740) and F1 (0.8721). fastai had the highest individual recall (0.8000). TabNet had the highest individual ROC-AUC (0.9764), while its PR-AUC was 0.6833. The old ensemble row has PR-AUC 0.8501, but it predates the fastai and TabNet runs and must be refreshed before drawing a final combined-model conclusion.

The regression metrics compare probabilities with binary labels and remain secondary. The framework scores shown below are drafts under the written rubric; the team should agree them before using them as final conclusions.

## Student-authored interpretation

Write the comparison and conclusion here in your own words. Check all numbers against the saved files.

Team check: Ask Person 2 to refresh the combined model after these new predictions are added; the existing ensemble files were left unchanged.

In [1]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "results").exists():
    ROOT = Path("/content/credit-card-fraud-detection")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git",
             str(ROOT)],
            check=True,
        )
os.chdir(ROOT)


In [2]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
PREDS = ROOT / "preds"
FIGURES.mkdir(exist_ok=True)

result_files = {
    path.stem: path for path in RESULTS.glob("*.json")
    if path.stem != "ensemble_members"
}
model_names = sorted(
    name for name in result_files
    if (PREDS / f"{name}_val.csv").exists() and (PREDS / f"{name}_test.csv").exists()
)
if not model_names:
    raise RuntimeError("No complete results and prediction pairs were found")
results = {name: json.loads(result_files[name].read_text()) for name in model_names}
test_predictions = {name: pd.read_csv(PREDS / f"{name}_test.csv") for name in model_names}
val_predictions = {name: pd.read_csv(PREDS / f"{name}_val.csv") for name in model_names}

reference = test_predictions[model_names[0]].sort_values("row_id").reset_index(drop=True)
for model_name in model_names[1:]:
    candidate = test_predictions[model_name].sort_values("row_id").reset_index(drop=True)
    if not np.array_equal(reference["row_id"], candidate["row_id"]):
        raise ValueError(f"Test row IDs do not align for {model_name}")
    if not np.array_equal(reference["y_true"], candidate["y_true"]):
        raise ValueError(f"Test labels do not align for {model_name}")

main_records = []
regression_records = []
for model_name in model_names:
    result = results[model_name]
    main_records.append({
        "Model": model_name,
        "Threshold": result["threshold"],
        "Precision": result["precision"],
        "Recall": result["recall"],
        "F1": result["f1"],
        "PR-AUC": result["pr_auc"],
        "ROC-AUC": result["roc_auc"],
        "Train (s)": result["train_seconds"],
        "Predict (s)": result["predict_seconds"],
    })
    regression_records.append({
        "Model": model_name,
        "RMSE": result["rmse"],
        "MAE": result["mae"],
        "R2": result["r2"],
    })

main_table = pd.DataFrame(main_records).sort_values("PR-AUC", ascending=False)
regression_table = pd.DataFrame(regression_records).sort_values("Model")
main_table.to_csv(RESULTS / "model_comparison_main.csv", index=False)
regression_table.to_csv(RESULTS / "model_comparison_regression.csv", index=False)
display(main_table.round(4))
display(regression_table.round(4))

def save_table_image(frame, path, title, digits=4):
    shown = frame.copy()
    for column in shown.select_dtypes(include=["number"]).columns:
        shown[column] = shown[column].map(lambda value: "" if pd.isna(value) else f"{value:.{digits}f}")
    width = max(10, 1.55 * len(shown.columns))
    fig, ax = plt.subplots(figsize=(width, 0.7 + 0.42 * (len(shown) + 1)))
    ax.axis("off")
    ax.set_title(title, pad=14)
    table = ax.table(
        cellText=shown.values,
        colLabels=shown.columns,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.4)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

save_table_image(main_table, FIGURES / "table_main.png", "Test classification metrics at validation-selected thresholds")
save_table_image(regression_table, FIGURES / "table_regression.png", "Probability regression metrics on the test split")

fig, ax = plt.subplots(figsize=(8, 6))
for model_name in model_names:
    frame = test_predictions[model_name]
    precision, recall, _ = precision_recall_curve(frame["y_true"], frame["y_prob"])
    ap = results[model_name]["pr_auc"]
    ax.plot(recall, precision, label=f"{model_name} (AP={ap:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Test precision-recall curves")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES / "pr_curves_all.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

ncols = 3
nrows = math.ceil(len(model_names) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.4 * nrows))
axes = np.asarray(axes).reshape(-1)
for ax, model_name in zip(axes, model_names):
    result = results[model_name]
    values = np.array([[result["tn"], result["fp"]], [result["fn"], result["tp"]]])
    ax.imshow(values, cmap="Blues")
    for (row, col), value in np.ndenumerate(values):
        ax.text(col, row, f"{value:,}", ha="center", va="center", color="black")
    ax.set_xticks([0, 1], labels=["Predicted 0", "Predicted 1"])
    ax.set_yticks([0, 1], labels=["Actual 0", "Actual 1"])
    ax.set_title(model_name)
for ax in axes[len(model_names):]:
    ax.axis("off")
fig.suptitle("Test confusion matrices at validation-selected thresholds")
fig.tight_layout()
fig.savefig(FIGURES / "confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

time_table = main_table.set_index("Model")[["Train (s)", "Predict (s)"]].copy()
for model_name in time_table.index:
    if model_name == "ensemble" or "member timings excluded" in str(results[model_name].get("notes", "")).lower():
        time_table.loc[model_name, ["Train (s)", "Predict (s)"]] = np.nan
ax = time_table.plot(kind="bar", figsize=(10, 5), color=["#147D83", "#C35A3A"])
ax.set_xlabel("Model")
ax.set_ylabel("Seconds recorded in result JSON")
ax.set_title("Recorded training and prediction time")
ax.legend(title="Stage")
plt.tight_layout()
plt.savefig(FIGURES / "time_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close()

best_ap = max(float(results[name]["pr_auc"]) for name in model_names)
transparency_draft = {
    "xgboost": (4, "gain_importance; SHAP_summary_and_case"),
    "fastai": (2, "permutation_importance_aggregate_only"),
    "tabnet": (5, "feature_importances_; per_row_explain_masks"),
    "keras": (2, "permutation_importance_aggregate_only"),
    "lightning": (2, "permutation_importance_aggregate_only"),
}
simplicity_draft = {
    "xgboost": (4, 14, "early_stopping; scale_pos_weight; automatic_device; requirements/xgboost.txt"),
    "fastai": (3, 37, "layers_64_32; dropout_0.2; weighted_CE; lr_find; early_stopping; requirements/fastai.txt"),
    "tabnet": (3, 40, "n_d_8; n_a_8; n_steps_3; weighted_CE; auc_early_stopping; requirements/tabnet.txt"),
    "keras": (3, 40, "dense_64_32; dropout; weighted_loss; early_stopping; requirements/keras.txt"),
    "lightning": (2, 65, "plain_Lightning_after_Flash_install_failure; custom_module; weighted_BCE; requirements/lightning.txt"),
}
score_records = []
for model_name in ["xgboost", "keras", "fastai", "tabnet", "lightning"]:
    if model_name not in results:
        score_records.append({"Model": model_name, "Status": "pending", "Simplicity": np.nan,
                             "Transparency": np.nan, "Efficacy": np.nan,
                             "Evidence": "Run the model notebook and review its evidence before scoring."})
        continue
    result = results[model_name]
    gap = best_ap - float(result["pr_auc"])
    efficacy = 5 if gap <= 0.02 else 4 if gap <= 0.05 else 3 if gap <= 0.10 else 2 if gap <= 0.20 else 1
    simple_score, train_predict_lines, simplicity_facts = simplicity_draft[model_name]
    transparency_score, transparency_method = transparency_draft[model_name]
    evidence = json.dumps({
        "train_predict_lines": train_predict_lines,
        "simplicity_facts": simplicity_facts,
        "train_seconds": result["train_seconds"],
        "predict_seconds": result["predict_seconds"],
        "transparency_method": transparency_method,
        "test_pr_auc_gap": gap,
        "precision": result["precision"],
        "recall": result["recall"],
    }, separators=(",", ":"))
    score_records.append({
        "Model": model_name,
        "Status": "draft; team review required",
        "Simplicity": simple_score,
        "Transparency": transparency_score,
        "Efficacy": efficacy,
        "Evidence": evidence,
    })

score_table = pd.DataFrame(score_records)
score_table.to_csv(RESULTS / "model_scores_draft.csv", index=False)
display(score_table)


Model  Threshold  Precision  Recall     F1  PR-AUC  ROC-AUC  Train (s)  Predict (s)
 ensemble     0.9277     0.9494  0.7895 0.8621  0.8488   0.9662     0.0000       0.0000
  xgboost     0.9394     0.9740  0.7895 0.8721  0.8382   0.9719     6.6759       0.0468
lightning     0.9999     0.8824  0.7895 0.8333  0.7576   0.9629     9.9909       0.0852
   fastai     0.9979     0.8837  0.8000 0.8398  0.7377   0.9612     6.0249       0.1422
    keras     0.9667     0.7957  0.7789 0.7872  0.7103   0.9628     1.2072       0.0199
   tabnet     0.9998     0.7976  0.7053 0.7486  0.6833   0.9764    45.5489       0.7191
    Model   RMSE    MAE       R2
 ensemble 0.0631 0.0181  -1.3815
   fastai 0.0869 0.0269  -3.5168
    keras 0.1708 0.1186 -16.4565
lightning 0.1072 0.0266  -5.8776
   tabnet 0.1769 0.0899 -17.7280
  xgboost 0.0203 0.0007   0.7529
    Model                      Status  Simplicity  Transparency  Efficacy                                                                                    